# מעבדה 11 — צברים סטטיסטיים

במעבדה הזו תגלו מאין בא גורם בולצמן, ותגלו זאת בלי לדמות שום דבר. מערכת קטנה חולקת קוונטי
אנרגיה עם אמבט של מתנדי איינשטיין; המחשב מונה כל דרך שבה המערכת המורכבת כולה יכולה להחזיק את
הקוונטים שלה, נותן לכל אחת מהן אותו משקל — הנחת היסוד של מודול 08 — ושואל באיזו תדירות המערכת
הקטנה נמצאת בכל אחד ממצביה.

המסלול: המודל והאובייקטים שלו (חלק 1); מערכת מורכבת קטנה מספיק כדי למנות אותה ביד (חלק 2);
ההתפלגות המשותפת השטוחה וההתפלגות השולית המעוקמת (חלק 3); הגדלת האמבט בטמפרטורה קבועה (חלק 4);
מדידת קצב ההתכנסות של האמבט הסופי, ושל האיבר שגזירת בולצמן משמיטה (חלק 5); קריאת הטמפרטורה מן
האמבט (חלק 6); של מי הטמפרטורה (חלק 7); המערכת הדו-רמתית בצורה סגורה (חלק 8); נספח רשות על
האטמוספרה (חלק 9); הבדיקות האוטומטיות (חלק 10); וחקירה חופשית (חלק 11).

עברו עליה בסדר. במקום שבו המחברת מבקשת מכם לנבא, רשמו את ניבויכם בתא המיועד לכך **לפני**
הרצת התא הבא. ניבוי שהתחייבתם אליו הוא הדרך האמינה היחידה לגלות שטעיתם.

## מפרט המודל

| | |
|---|---|
| **מערכת** | מערכת קטנה — מערכת דו-רמתית שהפער שלה מספר שלם של קוונטים, או מוצק איינשטיין של כמה מתנדים — החולקת $q_{\mathrm{tot}}$ קוונטים עם אמבט של מוצק איינשטיין בן $N_{\mathrm{bath}}$ מתנדים |
| **דינמיקה** | אין: כל מיקרו-מצב משותף נגיש נספר ומקבל משקל שווה; אין זמן, אין דגימה, אין מספרים אקראיים |
| **גבול** | המערכת המורכבת מבודדת, $q_{\mathrm{tot}}$ קבוע בדיוק; הדופן מעבירה קוונטים בלבד, ולכן שום עבודה אינה עוברת דרכה |
| **צבר** | מיקרו-קנוני בדיוק עבור המערכת המורכבת; ההתפלגות השולית של המערכת מתקרבת לצורה הקנונית ככל ש-$N_{\mathrm{bath}}$ גדל |
| **מוזנח** | אנרגיית הצימוד של הדופן, גדלי קוונט שונים, מבנה מרחבי, קצבי החלפה |
| **תקף כאשר** | המערכת והאמבט חולקים גודל קוונט אחד והמערכת המורכבת מבודדת; הצורה הקנונית דורשת גם $E_s \ll U_{\mathrm{bath}}$ |
| **אופני כישלון** | אמבט קטן מכדי שיהיה לו שיפוע $\ln\Omega$ חלק; $E_s$ הדומה בגודלה לאנרגיה הכוללת, שבה ההתפלגות השולית סוטה מן המעריכי |

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# thermolab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy, matplotlib and sympy, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# thermolab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("thermolab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from thermolab import ensembles
from thermolab.constants import AMU, K_B
from thermolab.validation import relative_error, scaling_exponent

# This module draws no random numbers at all: the enumeration is exact, so there is no
# generator to pass and nothing for a seed to change.

QUANTUM = 1.0e-21  # J, the size of one energy quantum, shared by system and bath
two_level = ensembles.two_level(1, QUANTUM)  # ground at 0, excited one quantum up
solid = ensembles.einstein_levels(3, 30, QUANTUM)  # three oscillators, levels k = 0..30
print(f"one quantum = {QUANTUM:.1e} J = {QUANTUM / K_B:.1f} K x k_B")
print(f"a bath with one quantum per oscillator sits at "
      f"T = {ensembles.bath_temperature(1.0, QUANTUM):.2f} K")

## חלק 1 — האובייקטים

**רשימת רמות** נותנת כל אחת מן האנרגיות של המערכת הקטנה, בקוונטים שלמים, וכמה מיקרו-מצבים
חולקים אותה. למערכת הדו-רמתית שתי רמות של מיקרו-מצב אחד כל אחת. המוצק בן שלושת המתנדים מחזיק
$k$ קוונטים ב-$g_k = \binom{k + 2}{k}$ דרכים — הכוכבים והמחיצות של מודול 01 — ולכן הרמות שלו
מנוונות.

האמבט הוא מוצק איינשטיין של $N_{\mathrm{bath}}$ מתנדים. כשהוא מחזיק $q$ קוונטים, יש לו
$\Omega_{\mathrm{bath}} = \binom{q + N_{\mathrm{bath}} - 1}{q}$ מיקרו-מצבים.

In [ ]:
print("two-level system:   quanta", two_level.quanta, "  degeneracy", two_level.degeneracy)
print("three-oscillator solid, first six levels:")
print("   quanta    ", solid.quanta[:6])
print("   degeneracy", np.rint(solid.degeneracy[:6]).astype(int))

### לנבא

התחייבו לתשובה לכל אחד מארבעת הניבויים של דף המודול לפני שתמשיכו — אותם המעבדה הזו בוחנת:

1. מערכת דו-רמתית נמצאת בשיווי משקל עם אמבט חום גדול. האם שני המצבים שלה שווי סבירות?
2. הגדילו את האמבט בטמפרטורה קבועה. האם ההיסטוגרמה של האכלוס משנה את *השיפוע* שלה, או רק
   נעשית *נקייה* יותר?
3. כל מיקרו-מצב *משותף* של המערכת והאמבט שווה הסתברות. מה סביר יותר: שהמערכת מעוררת, או
   שהיא במצב היסוד שלה?
4. חממו את האמבט ללא גבול. האם האוכלוסייה המעוררת שואפת ל-$1$, ל-$1/2$ או ל-$0$?

**הניבויים שלכם:**

1.
2.
3.
4.

## חלק 2 — מערכת מורכבת שאפשר למנות ביד

קחו את המערכת הדו-רמתית, אמבט של שלושה מתנדים, ושלושה קוונטים בסך הכול. לפני הרצת התא הבא,
מנו על נייר כל מיקרו-מצב משותף: באיזה מצב נמצאת המערכת, וכמה קוונטים מחזיק כל אחד משלושת
המתנדים של האמבט. כמה יש? בכמה מהם המערכת מעוררת?

**הספירה שלכם:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
states = ensembles.explicit_joint_microstates(two_level, n_bath=3, total_quanta=3)
names = ("ground ", "excited")
for level, _, bath in states:
    print(f"system {names[level]}   bath oscillators hold {bath}")

joint = ensembles.enumerate_joint(two_level, n_bath=3, total_quanta=3)
counts = np.rint(np.exp(joint.log_joint_count)).astype(int)
print(f"\n{len(states)} joint microstates; the formula counts {counts[0]} + {counts[1]}")
print(f"P(excited) = {counts[1]}/{counts.sum()} = {ensembles.marginal_occupation(joint)[1]:.4f}")

כל שורה למעלה היא מיקרו-מצב משותף אחד, ולכל אחד הסתברות $1/16$ — זו הנחת היסוד, ותו לא.
כשהמערכת מעוררת היא לקחה קוונט, והאמבט יכול להחזיק את השניים שנותרו רק
ב-$\binom{4}{2} = 6$ דרכים; במצב היסוד האמבט שומר את שלושתם, ב-$\binom{5}{3} = 10$ דרכים. המצבים
של המערכת משוקללים במה שהם משאירים לאמבט.

## חלק 3 — משותפת שטוחה, שולית מעוקמת

אותו טיעון, כטענת בדיקה. על אמבט של שישה מתנדים עם שישה קוונטים, לכל מיקרו-מצב משותף הסתברות
אחת בדיוק, ושני המצבים של המערכת עצמה אינם חולקים אותה.

In [ ]:
joint6 = ensembles.enumerate_joint(two_level, n_bath=6, total_quanta=6)
listed = ensembles.explicit_joint_microstates(two_level, n_bath=6, total_quanta=6)
per_joint = np.full(len(listed), joint6.joint_microstate_probability)
marginal6 = ensembles.marginal_occupation(joint6)

print(f"{len(listed)} joint microstates, each with probability 1/{len(listed)}")
print(f"distinct joint probabilities: {np.unique(per_joint).size}")
print(f"system marginal: P(ground) = {marginal6[0]:.4f},  P(excited) = {marginal6[1]:.4f}")

# The falsifying experiment for "the canonical ensemble means every microstate is equally
# probable": one probability for every joint microstate, two for the system's own states.
assert np.ptp(per_joint) == 0.0
assert abs(per_joint.sum() - 1.0) < 1e-12
assert marginal6[1] < marginal6[0]

## חלק 4 — הגדלת האמבט בטמפרטורה קבועה

השאירו קוונט אחד לכל מתנד באמבט, מה שמקבע את הטמפרטורה של האמבט, והגדילו את האמבט. עקבו אחר
אוכלוסיות הרמות של המוצק בן שלושת המתנדים מול ערכי בולצמן
$g_k e^{-E_k/(\kB T)}/Z$ של אמבט גדול לאין שיעור באותה טמפרטורה.

In [ ]:
temperature = ensembles.bath_temperature(1.0, QUANTUM)
reference = ensembles.boltzmann_distribution(solid, temperature)
sizes = [3, 10, 30, 100, 1000]
sweep = ensembles.bath_size_sweep(solid, sizes, quanta_per_oscillator=1.0)

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))
colours = plt.cm.viridis(np.linspace(0.0, 0.85, len(sizes)))
k = solid.quanta[:15]
for n_bath, marginal, colour in zip(sweep.n_bath, sweep.marginals, colours, strict=True):
    left.plot(k, marginal[:15], "o-", color=colour, ms=4, label=f"N_bath = {n_bath}")
    reach = marginal[:15] > 0
    right.plot(k[reach], np.log(marginal[:15][reach] / solid.degeneracy[:15][reach]), "o-",
               color=colour, ms=4)
left.plot(k, reference[:15], "k--", lw=2, label="Boltzmann")
right.plot(k, np.log(reference[:15] / solid.degeneracy[:15]), "k--", lw=2)
left.set_xlabel("E_k / quantum")
left.set_ylabel("P_k")
left.legend(fontsize=8)
right.set_xlabel("E_k / quantum")
right.set_ylabel("ln(P_k / g_k)")
right.set_title("log probability of one system microstate")
plt.tight_layout()
plt.show()

for n_bath, distance in zip(sweep.n_bath, sweep.distances, strict=True):
    print(f"N_bath = {n_bath:5d}   largest gap to Boltzmann = {distance:.2e}")

השיפוע של הלוח הימני הוא $-\beta\varepsilon$, והוא אינו משתנה כשהאמבט גדל — האנרגיה לכל מתנד
של האמבט קובעת אותו. מה שמשתנה הוא *העיקום*: הנקודות של אמבט קטן מתעקמות כלפי מטה, משום שכל
קוונט שהמערכת לוקחת מקרר מעט את האמבט ומייקר את הקוונט הבא. אמבט גדול אינו מרגיש בכך.

## חלק 5 — באיזו מהירות האמבט הסופי מתכנס?

### לנבא

הגזירה בדף המודול משמיטה איבר מסדר $E^2/(N_{\mathrm{bath}}(\kB T)^2)$.
אם תכפילו את האמבט בטמפרטורה קבועה, מה צריך לקרות לפער הגדול ביותר בין ההתפלגות השולית המדויקת
להתפלגות בולצמן?

**הניבוי שלכם:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
bath_sizes = [20, 40, 80, 160, 320, 640]
scaling = {}
for label, levels in (("two-level", two_level), ("3-oscillator solid", solid)):
    run = ensembles.bath_size_sweep(levels, bath_sizes, quanta_per_oscillator=1.0)
    (slope, _), cov = np.polyfit(np.log(run.n_bath), np.log(run.distances), 1, cov=True)
    scaling[label] = (run, slope, float(np.sqrt(cov[0, 0])))

fig, ax = plt.subplots(figsize=(6.5, 4))
for (label, (run, _, _)), colour in zip(scaling.items(), ("#2563eb", "#d97706"),
                                                strict=True):
    ax.loglog(run.n_bath, run.distances, "o", color=colour, label=label)
    ax.loglog(run.n_bath, run.distances[0] * run.n_bath[0] / run.n_bath, color=colour, lw=1)
ax.set_xlabel("N_bath")
ax.set_ylabel("max_k |P_k - P_k(Boltzmann)|")
ax.legend()
plt.tight_layout()
plt.show()

for label, (run, slope, error) in scaling.items():
    halvings = run.distances[:-1] / run.distances[1:]
    print(f"{label:<20} exponent alpha = {slope:.4f} +/- {error:.4f}   "
          f"(each doubling divides the gap by {halvings.min():.3f} to {halvings.max():.3f})")

דף המודול חוזה את האיבר המושמט במדויק. מעבר ל-$-\beta E_s$, $\ln P_s$ מקבל

$$
\frac{E_s^2}{2} \, \frac{\partial^2 \ln \Omega_{\mathrm{bath}}}{\partial U^2}
= -\frac{E_s^2}{2 \kB T^2 \, C_{\mathrm{bath}}} .
$$

הספירה המדויקת *מודדת* את אותו דבר, רמה אחר רמה, כ-

$$
\ln \frac{\Omega_{\mathrm{bath}}(q - k)}{\Omega_{\mathrm{bath}}(q)} + \beta E_k .
$$

השוו ביניהם.

In [ ]:
small_solid = ensembles.einstein_levels(3, 6, QUANTUM)
print(" N_bath    k   measured     predicted    ratio")
for n_bath in (30, 100, 300, 1000):
    joint_n = ensembles.enumerate_joint(small_solid, n_bath, n_bath)
    measured = ensembles.measured_log_correction(joint_n)
    predicted = ensembles.predicted_log_correction(joint_n)
    for level in (1, 4):
        print(f"{n_bath:6d}   {level:2d}   {measured[level]:+.5f}    {predicted[level]:+.5f}"
              f"    {measured[level] / predicted[level]:.4f}")

היחס שואף ל-$1$ ככל שהאמבט גדל, והמרחק שלו מ-$1$ הוא בעצמו בערך
$E_k/U_{\mathrm{bath}}$ — האיבר הבא של אותו פיתוח. "אמבט אינסופי" הוא קירוב שאת השגיאה שלו
מדדתם עכשיו פעמיים: פעם כפער בין התפלגויות, היורד כמו $1/N_{\mathrm{bath}}$, ופעם איבר אחר איבר.

## חלק 6 — קריאת הטמפרטורה מן האמבט

מודול 09 הגדיר טמפרטורה על ידי $1/T = \partial S/\partial U$. עם $S = \kB\ln\Omega$, ה-$\beta = 1/(\kB T)$
של האמבט הוא השיפוע של ה-$\ln\Omega$ שלו. מדדו אותו כפי שהייתם מודדים בעזרת הספירות בלבד: הוסיפו
קוונט, הוציאו קוונט, וחלקו בשני קוונטים. השוו לשיפוע המדויק,
$\beta\varepsilon = \sum_{j=1}^{N-1} 1/(q + j)$.

In [ ]:
print(" N_bath   beta (counted) [1/J]   beta (exact) [1/J]    relative gap   gap to ln2/quantum")
counted_gaps = []
for n_bath in (10, 20, 40, 80, 160, 320, 640):
    counted = ensembles.beta_of_bath(n_bath, n_bath, QUANTUM)
    exact = ensembles.bath_beta_exact(n_bath, n_bath, QUANTUM)
    counted_gaps.append(relative_error(counted, exact))
    print(f"{n_bath:6d}   {counted:.8e}      {exact:.8e}     {counted_gaps[-1]:.2e}"
          f"       {relative_error(exact, np.log(2) / QUANTUM):.2e}")

beta_order = -scaling_exponent([10, 20, 40, 80, 160, 320, 640], counted_gaps)
print(f"\nthe counted beta converges on the exact slope at order {beta_order:.2f} in 1/N_bath")
print(f"one oscillator alone: beta = {ensembles.bath_beta_exact(1, 5, QUANTUM)} "
      "(it holds any number of quanta in exactly one way)")

## חלק 7 — של מי הטמפרטורה?

### לנבא

שלושה אמבטים מחזיקים כל אחד בדיוק $100$ קוונטים, אבל יש בהם $25$, $100$ ו-$400$ מתנדים. אותה
מערכת דו-רמתית מוצבת על כל אחד בתורו. לאמבטים אותה אנרגיה. האם האכלוסים של המערכת יהיו זהים
בשלושתם?

**הניבוי שלכם:**

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
print(" N_bath   bath T [K]   P(excited)   Boltzmann at that T")
equal_energy = {}
for n_bath in (25, 100, 400):
    p = ensembles.marginal_occupation(ensembles.enumerate_joint(two_level, n_bath, 100))
    bath_t = 1.0 / (K_B * ensembles.bath_beta_exact(n_bath, 100, QUANTUM))
    boltzmann = ensembles.boltzmann_distribution(two_level, bath_t)[1]
    equal_energy[n_bath] = (p[1], boltzmann)
    print(f"{n_bath:6d}   {bath_t:9.1f}    {p[1]:.4f}       {boltzmann:.4f}")

אותה אנרגיה, שלוש תשובות שונות. האכלוס עוקב אחר *השיפוע* של האמבט, לא אחר האנרגיה שלו, וכל ערך
מתאים לגורם בולצמן בטמפרטורה של האמבט שלו. המערכת אינה מביאה טמפרטורה למגע; היא יורשת אחת. זו
הסיבה שמדחום קורא את הטמפרטורה של כל מה שהוא נוגע בו.

## חלק 8 — המערכת הדו-רמתית, בצורה סגורה

עבור אמבט גדול מספיק כדי להתייחס אליו כאינסופי, למערכת הדו-רמתית יש
$p_1 = 1/(e^{\Delta/(\kB T)} + 1)$ ו-$\langle E \rangle = \Delta\,p_1$.

In [ ]:
system = ensembles.TwoLevelSystem(gap=QUANTUM)
x = np.geomspace(0.05, 100.0, 300)  # k_B T / gap
temperatures = x * QUANTUM / K_B

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.6))
left.semilogx(x, system.ground_occupation(temperatures), color="#2563eb", label="p_0")
left.semilogx(x, system.excited_occupation(temperatures), color="#dc2626", label="p_1")
left.axhline(0.5, color="grey", ls="--", lw=1)
left.set_xlabel("k_B T / gap")
left.set_ylabel("occupation")
left.legend()
right.semilogx(x, system.mean_energy(temperatures) / QUANTUM, color="black")
right.axhline(0.5, color="grey", ls="--", lw=1)
right.set_xlabel("k_B T / gap")
right.set_ylabel("<E> / gap")
plt.tight_layout()
plt.show()

hottest = float(system.excited_occupation(temperatures[-1]))
print(f"coldest shown:  p_1 = {float(system.excited_occupation(temperatures[0])):.2e}")
print(f"hottest shown:  p_1 = {hottest:.6f}   <E>/gap = {hottest:.6f}")
print(f"largest p_1 anywhere on the curve: {np.max(system.excited_occupation(temperatures)):.6f}")

## חלק 9 — נספח רשות: האטמוספרה כגורם בולצמן

למולקולה בגובה $h$ יש אנרגיה $m g h$. התייחסו לשאר האטמוספרה כאל האמבט שלה, והצפיפות — ואיתה
הלחץ — יורדת כמו $e^{-m g h/(\kB T)}$, עם גובה סקאלה
$H = \kB T/(m g)$.

`data/11-standard-atmosphere.csv` היא האטמוספרה התקנית הבינלאומית; קראו את הכותרת שלה כדי לדעת
בדיוק מה היא: מודל ייחוס, איזותרמי ב-$216.65\ \mathrm{K}$ בין $11$ ל-$20\ \mathrm{km}$ ומתקרר
בהדרגה מתחת לכך. התאימו את השכבה האיזותרמית.

In [ ]:
from pathlib import Path

try:
    import piplite  # noqa: F401
except ImportError:
    # Desktop / nbmake: the repository root is three directories up from this notebook.
    csv_path = Path("..", "..", "..", "data", "11-standard-atmosphere.csv")
else:
    # JupyterLite bundles only the notebooks/ tree (jupyter_lite_config.json's
    # LiteBuildConfig.contents), so the browser gets its own copy of the CSV co-located here.
    csv_path = Path("data", "11-standard-atmosphere.csv")

altitude, air_t, air_p = np.genfromtxt(csv_path, delimiter=",", comments="#").T
upper = altitude >= 11_000
(h_slope, h_intercept), h_cov = np.polyfit(altitude[upper], np.log(air_p[upper]), 1, cov=True)
scale_height = -1.0 / h_slope
scale_error = np.sqrt(h_cov[0, 0]) / h_slope**2
g = 9.80665
molecule_mass = K_B * air_t[upper].mean() / (g * scale_height)

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(altitude / 1e3, np.log(air_p), "o", color="#2563eb", ms=4, label="standard atmosphere")
ax.plot(altitude / 1e3, h_intercept + h_slope * altitude, color="#dc2626",
        label="isothermal fit, 11-20 km")
ax.set_xlabel("h (km)")
ax.set_ylabel("ln(P / Pa)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"scale height of the isothermal layer: H = {scale_height:.1f} +/- {scale_error:.1e} m")
print("  (the error bar is only the table's rounding: a model atmosphere is exactly exponential)")
print(f"mean molecular mass from H: {molecule_mass / AMU:.3f} u  (air: 28.96 u)")
print(f"nitrogen at 288 K would have H = {K_B * 288 / (28.0134 * AMU * g) / 1e3:.2f} km")

מעל $11\ \mathrm{km}$ הנקודות יושבות על הקו, והשיפוע שלו מחזיר את המסה המולקולרית של
האוויר — גודל מיקרוסקופי מתוך פרופיל לחץ. מתחת ל-$11\ \mathrm{km}$ הן מתעקמות ממנו: האוויר שם
חם יותר, גובה הסקאלה שלו ארוך יותר, וגורם בולצמן יחיד בטמפרטורה אחת אינו מתאר שכבה שאינה
בטמפרטורה אחת. הכישלון הוא של ההנחה האיזותרמית, לא של גורם בולצמן. (זכרו מהי הטבלה הזאת: ההתאמה
אליה מחזירה את הקלטים של התקן עצמו. היא בוחנת את הקריאה שלכם בפיזיקה, לא את השמיים.)

## חלק 10 — בדיקות אוטומטיות

חישוב שלא בדקתם הוא תמונה, לא ראיה. אלה אותן טענות בדיקה שרצות בחבילת הבדיקות של הפרויקט.

In [ ]:
# 1. The hand count (Part 2): 16 joint microstates, 10 + 6, P(excited) = 6/16.
assert len(states) == 16 and list(counts) == [10, 6]
assert abs(ensembles.marginal_occupation(joint)[1] - 6 / 16) < 1e-14

# 2. The joint distribution is flat and the marginal is not (Part 3) -- checked above.

# 3. A large bath reproduces the Boltzmann distribution (Part 4) ...
assert sweep.distances[-1] < 2e-3
# ... and the finite-bath gap falls as 1/N_bath for both systems (Part 5).
for _, slope, _ in scaling.values():
    assert abs(slope + 1.0) < 0.02

# 4. The counted beta converges on the exact slope at second order (Part 6).
assert 1.95 < beta_order < 2.05

# 5. Equal energy, different baths, different occupations -- each Boltzmann at its own
#    temperature (Part 7).
occupations = [value[0] for value in equal_energy.values()]
assert occupations[0] > occupations[1] > occupations[2]
for exact_p, boltzmann_p in equal_energy.values():
    assert relative_error(exact_p, boltzmann_p) < 5e-3

# 6. The two-level system never inverts (Part 8).
assert np.all(system.excited_occupation(temperatures) < 0.5)

# 7. The isothermal layer's scale height gives back the mass of an air molecule (Part 9).
assert relative_error(molecule_mass / AMU, 28.9644) < 2e-3

print("all checks passed")

## חלק 11 — חקרו בעצמכם

בחרו את המערכת הקטנה, את הפער שלה, את גודל האמבט ואת מספר הקוונטים הממוצע שכל מתנד באמבט
מחזיק, ואז לחצו **Run Interact**. הלוח השמאלי מציג את האוכלוסיות המדויקות מול ערכי בולצמן; הלוח
הימני מציג את לוגריתם ההסתברות של מיקרו-מצב אחד של המערכת. שלושה דברים שכדאי לנסות:

1. אמבט של שניים או שלושה מתנדים באנרגיה נמוכה לכל מתנד. עד כמה רחוק מערכי בולצמן יכולות
   האוכלוסיות להיות?
2. פער של חמישה קוונטים על אמבט עם עשירית קוונט לכל מתנד. האם המצב המעורר נגיש בכלל על אמבט
   קטן?
3. כל הגדרה שהיא: האם תוכלו לגרום לרמה העליונה של מערכת דו-רמתית להיות המאוכלסת יותר?

In [ ]:
import ipywidgets as widgets


def explore(system_kind="two-level", gap_quanta=1, bath_oscillators=30, per_oscillator=1.0):
    levels = (ensembles.two_level(gap_quanta, QUANTUM) if system_kind == "two-level"
              else ensembles.einstein_levels(3, 30, QUANTUM))
    total = int(round(per_oscillator * bath_oscillators))
    joint_x = ensembles.enumerate_joint(levels, bath_oscillators, total)
    exact = ensembles.marginal_occupation(joint_x)
    boltzmann = ensembles.boltzmann_distribution(
        levels, ensembles.bath_temperature(per_oscillator, QUANTUM))
    shown = slice(0, 16)
    k = levels.quanta[shown]
    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.6))
    left.bar(k, exact[shown], width=0.6, color="#2563eb", label="exact, finite bath")
    left.plot(k, boltzmann[shown], "ko", label="Boltzmann")
    left.set_xlabel("E / quantum")
    left.set_ylabel("P")
    left.legend()
    reach = exact[shown] > 0
    right.plot(k[reach], np.log(exact[shown][reach] / levels.degeneracy[shown][reach]), "o-",
               color="#2563eb")
    right.plot(k, np.log(boltzmann[shown] / levels.degeneracy[shown]), "k--")
    right.set_xlabel("E / quantum")
    right.set_ylabel("ln(P / g)")
    plt.tight_layout()
    plt.show()
    print(f"q_tot = {total} quanta;  largest gap to Boltzmann = "
          f"{ensembles.sup_norm(exact, boltzmann):.2e}")


widgets.interact_manual(
    explore,
    system_kind=["two-level", "3-oscillator solid"],
    gap_quanta=widgets.IntSlider(value=1, min=1, max=5),
    bath_oscillators=widgets.IntSlider(value=30, min=2, max=500),
    per_oscillator=widgets.FloatLogSlider(value=1.0, base=10, min=-1, max=1, step=0.05),
)

## בחנו את הבנתכם

הריצו את התא שלהלן לחידון הנבדק אוטומטית. אותן שאלות, עם הסברים כתובים לכל אפשרות, נמצאות
בדף המודול.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "11-ensembles.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## לפני שאתם עוזבים

רשמו כמה משפטים על כל אחת, בתא שלהלן.

1. מה ניבאתם שהתברר כשגוי, ומה בדיוק היה הפגם בנימוקכם?
2. בחלק 3 לכל מיקרו-מצב משותף הייתה אותה הסתברות ולשני המצבים של המערכת לא. הסבירו, בלי
   משוואות, כיצד שני הדברים יכולים להיות נכונים.
3. לשלושת האמבטים של חלק 7 הייתה אותה אנרגיה. אמרו בדיוק לאיזו תכונה של אמבט המערכת מגיבה,
   ומדוע.
4. שום דבר במעבדה הזו לא זז, לא התנגש ולא השתמש במספר אקראי. מה, אם כן, יצר את גורם בולצמן?

**התשובות שלכם:**

1.
2.
3.
4.